# Treaty Pricing Submission Analyst Agent

A working reference implementation of the agent, structured exactly around
the brief: 8 governed tools, 5 RAG documents, and 4 enforced guardrails.

## Files

| File | Role |
|----|----|
| `rag_documents/` | The 5 governed documents (methodology, approved trend assumptions, expense/capital parameters, underwriting guidelines, peer-review checklist). These are the only source of truth for assumptions. |
| `rag_store.py` | Loads the documents and exposes their key parameters (trends, credibility standard, expense/capital loads) as structured constants the tools/guardrails can check against. |
| `tools.py` | The 8 governed tools (`validate_pricing_submission`, `trend_and_onlevel_claims`, `apply_treaty_structure_to_losses`, `calculate_burning_cost`, `fit_frequency_severity_models`, `calculate_credibility_weight`, `calculate_technical_price`, `run_pricing_sensitivities`). Every call is appended to a shared `TRACE` log. |
| `guardrails.py` | Enforces the 4 guardrails in code (raises `GuardrailViolation` on breach), not just in a prompt. |
| `agent.py` | Orchestrator (`TreatyPricingAgent`) that runs the pipeline in methodology order, runs sensitivities, drafts the memo, and runs guardrail checks before releasing it. |
| `run_demo.py` | End-to-end demo on a synthetic 11-year property-cat wind submission. |
| `pricing_review_note.md` | The memo produced by that demo run. |

## Run it

    python3 run_demo.py

## How each guardrail is enforced

1.  **Cannot select an unapproved assumption** — `tools.py` never hardcodes a
    trend/credibility/expense value itself; it pulls from `rag_store.py`, and
    `guardrails.assert_approved_trend` / `assert_approved_tiv_trend` /
    `assert_approved_expense_capital_margin` / `assert_credibility_standard`
    raise if anything doesn’t match the governed document’s constants exactly.
2.  **Cannot quote, bind, or recommend a final commercial price** —
    `guardrails.assert_no_commercial_recommendation` scans the drafted memo
    for commercial-decision language before it’s released, and the memo
    template itself only ever states a “Technical Price Indication.”
3.  **Technical calculation and commercial decision must be clearly
    separated** — `guardrails.assert_sections_separated` requires the memo to
    contain `SECTION A: TECHNICAL PRICE INDICATION` followed by
    `SECTION B: COMMERCIAL DECISION`, with Section B left as
    `[to be determined by underwriting management]`.
4.  **Every numeric result must be reproducible from the tool trace** —
    `tools.TRACE` logs every tool call’s inputs/outputs; the memo prints the
    full trace as an appendix; and `guardrails.assert_trace_covers_numbers`
    checks (as a build-time test, not just documentation) that every number
    quoted in the memo body actually traces back to a value the tools
    produced.

## Extending it

- Swap `rag_store.py`’s constants for a real vector-store lookup against the
  governed documents once those live in an actual document repository.
- Replace the simplified lognormal method-of-moments severity fit in
  `fit_frequency_severity_models` with a call to an internal cat model /
  actuarial library if available — the tool boundary is already isolated so
  this is a drop-in change.
- Wire `validate_pricing_submission`’s issue list into a required
  human-acknowledgment step before the pipeline proceeds, if you want a
  stricter “PASS_WITH_FLAGS still needs sign-off” gate.